In [4]:
%pip install pybullet

     ---------------------------------------- 0.0/80.5 MB ? eta -:--:--
     - -------------------------------------- 2.1/80.5 MB 10.7 MB/s eta 0:00:08
     -- ------------------------------------- 4.5/80.5 MB 11.2 MB/s eta 0:00:07
     --- ------------------------------------ 6.8/80.5 MB 11.0 MB/s eta 0:00:07
     ---- ----------------------------------- 9.4/80.5 MB 11.3 MB/s eta 0:00:07
     ----- --------------------------------- 11.5/80.5 MB 10.9 MB/s eta 0:00:07
     ------ -------------------------------- 13.9/80.5 MB 10.9 MB/s eta 0:00:07
     -------- ------------------------------ 16.5/80.5 MB 11.1 MB/s eta 0:00:06
     --------- ----------------------------- 18.9/80.5 MB 11.1 MB/s eta 0:00:06
     ---------- ---------------------------- 21.2/80.5 MB 11.1 MB/s eta 0:00:06
     ----------- --------------------------- 23.6/80.5 MB 11.1 MB/s eta 0:00:06
     ------------ -------------------------- 26.2/80.5 MB 11.2 MB/s eta 0:00:05
     ------------- ------------------------- 28.


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import pybullet as p
import pybullet_data
import math
import pandas as pd

# 1. PyBullet 물리 엔진 초기화 (GUI 창 없이 계산만 빠르게 수행)
physicsClient = p.connect(p.DIRECT) 
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.setGravity(0, 0, -9.81)

# 바닥 생성
planeId = p.loadURDF("plane.urdf")

# 2. 물리적 고정값 설정
FIXED_FORCE = 3.6        # 발사 힘 (N·s)
BALL_RADIUS = 0.045 / 2  # 지름 4.5cm -> 반지름 0.0225m

# 3. 실험 변수 범위 설정 (총 2 x 5 x 13 = 130개 데이터 세트)
ball_masses = [0.003, 0.027, 0.0045]                            # 무게: 3g, 27g (kg 단위)
drags = [0.0, 0.05, 0.1, 0.15, 0.2]                     # 공기 저항 계수 (kg/s)
angles = list(range(0, 46, 1))                         # 각도: 15° ~ 75° (5° 단위)

raw_results = []

# 4. 시뮬레이션 반복 루프 실행
for mass in ball_masses:
    for drag in drags:
        for angle in angles:
            
            # 매 실험마다 공 새로 생성 (위치 X=0, Y=0, Z=1)
            ball = p.createMultiBody(
                baseMass=mass,
                baseCollisionShapeIndex=p.createCollisionShape(p.GEOM_SPHERE, radius=BALL_RADIUS),
                basePosition=[0, 0, 1]
            )
            
            # 삼각함수를 이용한 발사 벡터(X, Z) 계산
            angle_rad = math.radians(angle)
            force_x = FIXED_FORCE * math.cos(angle_rad)
            force_z = FIXED_FORCE * math.sin(angle_rad)
            
            # 공에 순간적인 충격량(발사 힘) 부여
            p.applyExternalForce(ball, -1, [force_x, 0, force_z], [0, 0, 0], p.LINK_FRAME)
            
            # 실시간 물리 연산 루프 (바닥에 닿으면 강제 탈출)
            for _ in range(2000):
                # 실시간 속도 벡터를 기반으로 반대 방향의 공기 저항(항력) 적용
                velocity, _ = p.getBaseVelocity(ball)
                drag_force = [-drag * v for v in velocity]
                p.applyExternalForce(ball, -1, drag_force, [0, 0, 0], p.LINK_FRAME)
                
                p.stepSimulation()
                
                # 공의 현재 좌표 확인
                ball_pos, _ = p.getBasePositionAndOrientation(ball)
                if ball_pos[2] <= BALL_RADIUS:  # 중심 높이가 반지름 이하 = 바닥 착지 완료
                    break
                    
            # 바닥에 도달한 최종 수평 이동 거리(X좌표)
            final_distance = ball_pos[0]
            
            # 다음 실험을 위해 사용한 오브젝트 제거
            p.removeBody(ball)
            
            # 데이터 수집 리스트에 추가
            raw_results.append({
                'Weight_kg': mass,
                'Air_Resistance_kgs': drag,
                'Angle_deg': angle,
                'Landing_Distance_m': round(final_distance, 3)
            })

# 5. 물리 엔진 안전하게 종료
p.disconnect()

# 6. Pandas 데이터프레임 변환 및 단위 컬럼명 지정
df = pd.DataFrame(raw_results)
df.columns = [
    'Weight(kg)', 
    'Air_Resistance(kg/s)', 
    'Angle(deg)', 
    'Landing_Distance(m)'
]

# 7. 인덱스를 제외하고 깔끔하게 CSV로 저장
file_name = "cannon_experiment1_data.csv"
df.to_csv(file_name, index=False, encoding='utf-8-sig')

print(f"🎉 CSV 파일 저장 완료! 총 {len(df)}개 조건의 데이터가 '{file_name}'로 저장되었습니다.")

🎉 CSV 파일 저장 완료! 총 690개 조건의 데이터가 'cannon_experiment1_data.csv'로 저장되었습니다.
